# 02 · Per-beam NBI power (addendum)

The first extraction only kept **total** neutral-beam power. This notebook adds absorbed power (`PABS`) for each of the **8 beams** individually (peak + mean → 16 new columns, 104 total).

Includes a data-length guard: ~15 shots had a beam whose `data` and `dim0` arrays disagreed in length (e.g. 131,072 samples vs. 2 timestamps). Instead of crashing the shot, only that beam is skipped.

**Result:** 3,718 / 3,718 shots succeeded, 0 failed.

In [ ]:
import h5py, os, numpy as np, pandas as pd, csv

data_dir = "/path/to/big_d3d_data/d3d_time_series_data"
param_csv_path = "/path/to/outputs/shot_parameters_v2.csv"

df = pd.read_csv(param_csv_path)
shot_list = sorted(df["shot_number"].tolist())

NBI_BEAMS = ["NB15L", "NB15R", "NB21L", "NB21R", "NB30L", "NB30R", "NB33L", "NB33R"]

print("Shots to process:", len(shot_list))

In [ ]:
def get_nbi_power_by_beam(shot_number, f, tmin=0, tmax=6000):
    """
    Per-beam PABS (absorbed NBI power), one peak/mean pair per beam.
    Guards against data/time length mismatches (seen on ~15 shots, e.g.
    146029, where data had 131072 points but dim0 only had 2) - skips
    just that beam rather than crashing the whole shot's extraction.
    """
    results = {}
    for beam in NBI_BEAMS:
        key = f"\\D3D::TOP.NB.{beam}:PABS"
        try:
            if key not in f[str(shot_number)]["D3D"]:
                results[f"peak_pabs_{beam}"] = None
                results[f"mean_pabs_{beam}"] = None
                continue

            diag = f[str(shot_number)]["D3D"][key]
            data = diag["data"][:]
            time = diag["dim0"][:]

            if len(data) != len(time):
                results[f"peak_pabs_{beam}"] = None
                results[f"mean_pabs_{beam}"] = None
                continue

            mask = (time >= tmin) & (time <= tmax)
            data_masked = data[mask]
            nz = data_masked[data_masked > 0]

            if len(nz) == 0:
                results[f"peak_pabs_{beam}"] = None
                results[f"mean_pabs_{beam}"] = None
            else:
                results[f"peak_pabs_{beam}"] = float(np.percentile(nz, 95))
                results[f"mean_pabs_{beam}"] = float(np.mean(nz))
        except KeyError:
            results[f"peak_pabs_{beam}"] = None
            results[f"mean_pabs_{beam}"] = None
    return results


def extract_per_beam_nbi(shot_number, data_dir):
    filename = os.path.join(data_dir, f"{shot_number}.h5")
    if not os.path.exists(filename):
        return None
    try:
        with h5py.File(filename, "r") as f:
            if str(shot_number) not in f:
                return None
            result = {"shot_number": shot_number}
            result.update(get_nbi_power_by_beam(shot_number, f))
            return result
    except Exception as e:
        print(f"Error on shot {shot_number}: {e}")
        return None

print("Per-beam extraction function defined")

In [ ]:
for test_shot in [144187, 144229, 146029]:
    result = extract_per_beam_nbi(test_shot, data_dir)
    print(f"Shot {test_shot}:")
    if result is None:
        print("  FAILED - returned None")
    else:
        for k, v in result.items():
            print(f"  {k}: {v}")
    print()

In [ ]:
fieldnames = ["shot_number"] + [f"{s}_pabs_{b}" for b in NBI_BEAMS for s in ("peak", "mean")]

output_file = "/path/to/outputs/shot_nbi_by_beam.csv"
total, success, failed = len(shot_list), 0, 0

with open(output_file, "w", newline="") as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    for i, shot_number in enumerate(shot_list):
        if i % 500 == 0:
            print("Progress:", i, "/", total, "| succeeded:", success, "| failed:", failed)
        result = extract_per_beam_nbi(shot_number, data_dir)
        if result is None:
            failed += 1
            continue
        writer.writerow(result)
        success += 1

print("Succeeded:", success, "| Failed:", failed)
print("Saved to:", output_file)

In [ ]:
df_beams = pd.read_csv(output_file)
df_merged = df.merge(df_beams, on="shot_number", how="left")

merged_output = "/path/to/outputs/shot_parameters_v3.csv"
df_merged.to_csv(merged_output, index=False)

print("Merged shape:", df_merged.shape)
print("Saved to:", merged_output)